# NullVector Real PDF Parser + Tree Demo Notebook
## v2 Acquisition + Tree Manual Demo
Purpose: exercise the current acquisition/projection runtime and deterministic tree pipeline against the local real document `903000608.pdf`.


### Environment Assumptions
- This notebook is a local operator demo and is not CI acceptance evidence.
- It requires `903000608.pdf` at the repository root.
- It prefers existing local acquisition artifacts under `notebooks/artifacts/acquisition_runs/` when they match the real PDF fingerprint.
- It builds the current acquisition-backed tree path and does not depend on the deprecated parse-manifest workflow.


In [ ]:
# environment setup
from pathlib import Path

REPO_ROOT = Path.cwd()
PDF_PATH = REPO_ROOT / "903000608.pdf"
LOCAL_ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "artifacts" / "acquisition_runs"
EXECUTION_ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "spec_v1_parser_tree_demo"
LOCAL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
EXECUTION_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        "Real demo notebook requires 903000608.pdf at the repository root."
    )

print(f"repo_root={REPO_ROOT}")
print(f"pdf_path={PDF_PATH}")
print(f"local_artifact_root={LOCAL_ARTIFACT_ROOT}")
print(f"execution_artifact_root={EXECUTION_ARTIFACT_ROOT}")


In [ ]:
# imports
import json
import shutil
from typing import Any

from nullvector.domain import AcquisitionRequest, AcquisitionSettings, TreeBuildRequest
from nullvector.ingest import acquire_document
from nullvector.ingest.acquisition_artifacts import settings_digest
from nullvector.ingest.fingerprint import fingerprint_document
from nullvector.tree import TreeConflictError, build_tree


In [ ]:
# configuration
PDF_FINGERPRINT = fingerprint_document(str(PDF_PATH))
SETTINGS_DIGEST = settings_digest(AcquisitionSettings())
RUN_SUFFIX = f"{PDF_FINGERPRINT.sha256[:8]}-{SETTINGS_DIGEST[:8]}"
ACQUISITION_RUN_ID = f"spec_v1_acquisition-{RUN_SUFFIX}"
TREE_RUN_ID = f"spec_v1_tree-{RUN_SUFFIX}"
REPRESENTATIVE_PAGE_INDEXES = (11, 23, 32, 34, 35)
LOCAL_MANIFEST_EXPECTED_RELATIVE_PATHS = {
    "ledger_path": "ledger/canonical-document-ledger.json",
    "selected_outline_path": "outline/selected.json",
    "source_copy_path": "source/original.pdf",
    "pymupdf_outline_path": "outline/pymupdf.normalized.json",
    "pymupdf_rich_outline_path": "outline/pymupdf.rich.json",
    "pypdf_outline_path": "outline/pypdf.normalized.json",
    "projection_view_path": "projection/tree-synthesis-view.json",
}


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def resolve_artifact_path(acquisition_root: Path, stored_path: str | None) -> Path | None:
    if stored_path is None:
        return None
    path = Path(stored_path)
    if path.is_absolute():
        return path
    if path.exists():
        return path.resolve()
    return acquisition_root / path


def manifest_artifacts_are_usable(manifest_path: Path, payload: dict[str, Any]) -> bool:
    acquisition_root = manifest_path.parent
    for key in LOCAL_MANIFEST_EXPECTED_RELATIVE_PATHS:
        resolved = resolve_artifact_path(acquisition_root, payload.get(key))
        if resolved is None or not resolved.exists():
            return False
    return True


def normalize_local_manifest_payload(
    manifest_path: Path,
    payload: dict[str, Any],
) -> dict[str, Any]:
    acquisition_root = manifest_path.parent
    normalized = dict(payload)
    normalized["artifact_root"] = str(acquisition_root)
    for key, relative_path in LOCAL_MANIFEST_EXPECTED_RELATIVE_PATHS.items():
        expected = acquisition_root / relative_path
        if expected.exists():
            normalized[key] = relative_path
    return normalized


def resolve_existing_acquisition_manifest(document_id: str, sha256: str) -> tuple[Path | None, str]:
    candidate = LOCAL_ARTIFACT_ROOT / ACQUISITION_RUN_ID / document_id / "manifest.json"
    if not candidate.exists():
        return None, "missing_local_manifest"
    payload = load_json(candidate)
    if payload.get("source_fingerprint", {}).get("sha256") != sha256:
        return None, "mismatched_local_manifest"
    if manifest_artifacts_are_usable(candidate, payload):
        return candidate, "reused_local_manifest"

    normalized = normalize_local_manifest_payload(candidate, payload)
    if manifest_artifacts_are_usable(candidate, normalized):
        candidate.write_text(
            json.dumps(normalized, indent=2, sort_keys=True, ensure_ascii=True),
            encoding="utf-8",
        )
        return candidate, "reused_local_manifest"
    return None, "stale_local_manifest"


def ensure_acquisition_manifest() -> tuple[dict[str, Any], Path, str]:
    manifest_path, mode = resolve_existing_acquisition_manifest(
        PDF_FINGERPRINT.document_id,
        PDF_FINGERPRINT.sha256,
    )
    if manifest_path is not None:
        return load_json(manifest_path), manifest_path, mode

    acquisition_run_root = LOCAL_ARTIFACT_ROOT / ACQUISITION_RUN_ID
    if acquisition_run_root.exists():
        shutil.rmtree(acquisition_run_root)

    manifest = acquire_document(
        AcquisitionRequest(
            source_path=str(PDF_PATH),
            acquisition_run_id=ACQUISITION_RUN_ID,
            artifact_root=str(LOCAL_ARTIFACT_ROOT),
        )
    )
    manifest_path = Path(manifest.artifact_root) / "manifest.json"
    return load_json(manifest_path), manifest_path, "acquired_fresh"


In [ ]:
# execution
acquisition_manifest, acquisition_manifest_path, acquisition_mode = ensure_acquisition_manifest()
acquisition_root = acquisition_manifest_path.parent
selected_outline_path = resolve_artifact_path(
    acquisition_root, acquisition_manifest["selected_outline_path"]
)
ledger_path = resolve_artifact_path(acquisition_root, acquisition_manifest["ledger_path"])
projection_path = resolve_artifact_path(
    acquisition_root, acquisition_manifest["projection_view_path"]
)
if selected_outline_path is None or ledger_path is None or projection_path is None:
    raise RuntimeError("Resolved acquisition manifest is missing required artifact paths.")

selected_outline = load_json(selected_outline_path)
canonical_ledger = load_json(ledger_path)
projection_view = load_json(projection_path)
pdf_fingerprint = acquisition_manifest["source_fingerprint"]

tree_manifest = None
for candidate_tree_run_id in (
    TREE_RUN_ID,
    f"{TREE_RUN_ID}-strategy-v2",
    f"{TREE_RUN_ID}-{pdf_fingerprint['sha256'][:8]}-major-changes-v2",
):
    try:
        tree_manifest = build_tree(
            TreeBuildRequest(
                acquisition_manifest_path=str(acquisition_manifest_path),
                tree_run_id=candidate_tree_run_id,
            )
        )
        break
    except TreeConflictError:
        continue
if tree_manifest is None:
    raise RuntimeError("Unable to resolve a reusable local tree_run_id for the demo notebook.")
tree_manifest_path = Path(tree_manifest.artifact_root) / "manifest.json"
build_report = load_json(Path(tree_manifest.build_report_path))
verification_report = load_json(Path(tree_manifest.verification_report_path))
node_cards = load_json(Path(tree_manifest.node_cards_path))
committed_nodes = load_json(Path(tree_manifest.committed_hierarchy_path))
unassigned_spans = load_json(Path(tree_manifest.unassigned_spans_path))


In [ ]:
# execution
ledger_pages_by_index = {page["page_index"]: page for page in canonical_ledger["pages"]}
projection_pages_by_index = {page["page_index"]: page for page in projection_view["pages"]}
representative_pages = []
for page_index in REPRESENTATIVE_PAGE_INDEXES:
    ledger_page = ledger_pages_by_index.get(page_index)
    projection_page = projection_pages_by_index.get(page_index)
    if ledger_page is None:
        continue
    line_preview = [
        block["content"]
        for block in ledger_page["blocks"]
        if block["block_type"] == "line_block"
    ][:3]
    representative_pages.append(
        {
            "page_index": page_index,
            "page_label": ledger_page["page_label"],
            "native_available": ledger_page["native_available"],
            "block_count": len(ledger_page["blocks"]),
            "line_preview": line_preview,
            "projection_line_count": len(projection_page["lines"]) if projection_page else 0,
            "unresolved_region_count": (
                len(projection_page["unresolved_regions"]) if projection_page else 0
            ),
        }
    )

top_level_node_cards = [card for card in node_cards if card["level"] == 1][:10]
failed_node_results = [
    {
        "subject_id": result["subject_id"],
        "status": result["status"],
        "issue_codes": [issue["code"] for issue in result.get("issues", [])],
    }
    for result in verification_report["node_results"]
    if result["status"] != "passed"
][:5]


In [ ]:
# inspect results
summary = {
    "pdf": {
        "path": str(PDF_PATH),
        "sha256": pdf_fingerprint["sha256"],
        "page_count": pdf_fingerprint["page_count"],
        "acquisition_mode": acquisition_mode,
    },
    "acquisition": {
        "manifest_path": str(acquisition_manifest_path),
        "outline_source": acquisition_manifest["selected_outline_source"],
        "page_count": acquisition_manifest["page_count"],
        "projection_page_count": len(projection_view["pages"]),
        "selected_outline_entries": len(selected_outline["entries"]),
    },
    "outline_preview": selected_outline["entries"][:6],
    "representative_pages": representative_pages,
    "tree": {
        "manifest_path": str(tree_manifest_path),
        "committed_node_count": tree_manifest.committed_node_count,
        "unassigned_span_count": tree_manifest.unassigned_span_count,
        "verification_status": verification_report["status"],
        "outline_trust_mode": build_report["outline_trust_mode"],
    },
    "top_level_node_cards": [
        {
            "title": card["title"],
            "level": card["level"],
            "page_span": card["page_span"],
        }
        for card in top_level_node_cards
    ],
    "failed_node_results": failed_node_results,
    "unassigned_spans_preview": unassigned_spans[:5],
}
print(json.dumps(summary, indent=2, sort_keys=True))


### Known Limitations
- This notebook depends on a local real PDF and local notebook artifact state, so it is not part of mandatory CI acceptance.
- It now exercises the acquisition/projection tree path; the deprecated parse-manifest workflow is intentionally not used here.
- Verification quality on a large real PDF still depends on the current deterministic hierarchy logic and may expose unresolved structure honestly.
